In [ ]:
# ── Summary Table ──
print("=" * 70)
print("EVALUATION SUMMARY")
print("=" * 70)
print(f"  Positive pairs analyzed: {len(positive_pairs)}")
print(f"  Negative pairs analyzed: {len(negative_pairs)}")
print(f"  Total examples:          {len(all_rows)}")
print()
print("  Success Criteria:")
print(f"    SC1 (leading across tasks): {'MET' if block6_results['eval_sc1_met'] > 0.5 else 'NOT MET'}")
print(f"    SC2 (variance scaling):     {'MET' if block6_results['eval_sc2_met'] > 0.5 else 'NOT MET'}")
print(f"    SC3 (classifier gain):      {'MET' if block6_results['eval_sc3_met'] > 0.5 else 'NOT MET'}")
print()
print(f"  Hypothesis: {verdict}")
print("=" * 70)

# ── Visualization: CSD indicators vs difficulty for each positive pair ──
fig, axes = plt.subplots(len(positive_pairs), 3, figsize=(14, 4 * len(positive_pairs)),
                          squeeze=False, constrained_layout=True)
fig.suptitle("CSD Indicators vs Difficulty Level (Positive Pairs)", fontsize=14, fontweight="bold")

for i, pair in enumerate(positive_pairs):
    rows = sorted(pair["rows"], key=lambda x: x["difficulty_level"])
    levels = [r["difficulty_level"] for r in rows]
    d_star = pair["d_star"]
    title_prefix = f"{pair['task']} x {pair['smodel']}"

    # Panel 1: Accuracy + Embedding Variance
    ax1 = axes[i][0]
    acc = [r["accuracy"] for r in rows]
    var = [r["embedding_variance"] for r in rows]
    ax1.plot(levels, acc, "o-", color="tab:blue", label="Accuracy", linewidth=2)
    ax1_twin = ax1.twinx()
    ax1_twin.plot(levels, var, "s--", color="tab:red", label="Emb. Variance", linewidth=1.5, alpha=0.8)
    ax1.axvline(d_star, color="gray", linestyle=":", alpha=0.7, label=f"d*={d_star}")
    ax1.set_xlabel("Difficulty Level")
    ax1.set_ylabel("Accuracy", color="tab:blue")
    ax1_twin.set_ylabel("Variance", color="tab:red")
    ax1.set_title(f"{title_prefix}\nAccuracy & Variance")
    ax1.legend(loc="lower left", fontsize=8)

    # Panel 2: Bimodality indicators
    ax2 = axes[i][1]
    bc = [r["bimodality_coefficient"] for r in rows]
    sil = [r["silhouette_k2"] for r in rows]
    ax2.plot(levels, bc, "^-", color="tab:green", label="Bimodality Coeff", linewidth=1.5)
    ax2.plot(levels, sil, "v-", color="tab:purple", label="Silhouette K=2", linewidth=1.5)
    ax2.axhline(0.555, color="tab:green", linestyle=":", alpha=0.5, label="BC threshold")
    ax2.axhline(0.3, color="tab:purple", linestyle=":", alpha=0.5, label="Sil threshold")
    ax2.axvline(d_star, color="gray", linestyle=":", alpha=0.7)
    ax2.set_xlabel("Difficulty Level")
    ax2.set_ylabel("Indicator Value")
    ax2.set_title(f"{title_prefix}\nBimodality Indicators")
    ax2.legend(fontsize=7, loc="best")

    # Panel 3: Dip test + Disagreement
    ax3 = axes[i][2]
    dip_p = [r["dip_pvalue"] for r in rows]
    disagr = [r["disagreement_rate"] for r in rows]
    ax3.semilogy(levels, [max(p, 1e-6) for p in dip_p], "D-", color="tab:orange",
                 label="Dip p-value", linewidth=1.5)
    ax3.axhline(0.05, color="tab:orange", linestyle=":", alpha=0.5, label="p=0.05")
    ax3_twin = ax3.twinx()
    ax3_twin.plot(levels, disagr, "x-", color="tab:cyan", label="Disagreement", linewidth=1.5)
    ax3.axvline(d_star, color="gray", linestyle=":", alpha=0.7)
    ax3.set_xlabel("Difficulty Level")
    ax3.set_ylabel("Dip p-value (log)", color="tab:orange")
    ax3_twin.set_ylabel("Disagreement", color="tab:cyan")
    ax3.set_title(f"{title_prefix}\nDip Test & Disagreement")
    ax3.legend(loc="upper left", fontsize=7)
    ax3_twin.legend(loc="upper right", fontsize=7)

plt.savefig("csd_indicators.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved csd_indicators.png")

## Results Visualization

Summary of key evaluation metrics across all analysis blocks, plus CSD indicator trajectories for positive pairs showing the relationship between difficulty level, accuracy, and CSD signals.

In [ ]:
def compute_classifier_f1(pair):
    """Build logistic regression classifier for near-boundary detection. Returns (csd_f1, baseline_f1)."""
    rows = pair["rows"]
    d_star = pair["d_star"]
    X_csd, X_base, y = [], [], []
    for r in rows:
        feats_csd = [
            r["dip_statistic"] if not np.isnan(r["dip_statistic"]) else 0,
            r["silhouette_k2"] if not np.isnan(r["silhouette_k2"]) else 0,
            r["bimodality_coefficient"] if not np.isnan(r["bimodality_coefficient"]) else 0,
            r["embedding_variance"] if not np.isnan(r["embedding_variance"]) else 0,
        ]
        feats_base = [r["disagreement_rate"] if not np.isnan(r["disagreement_rate"]) else 0]
        label = 1 if abs(r["difficulty_level"] - d_star) <= 2 else 0
        X_csd.append(feats_csd)
        X_base.append(feats_base)
        y.append(label)

    X_csd, X_base, y = np.array(X_csd), np.array(X_base), np.array(y)
    if len(np.unique(y)) < 2 or len(y) < 4:
        return float("nan"), float("nan")

    loo = LeaveOneOut()
    preds_csd = np.zeros(len(y))
    preds_base = np.zeros(len(y))

    for train_idx, test_idx in loo.split(X_csd):
        try:
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_csd[train_idx])
            X_test = scaler.transform(X_csd[test_idx])
            clf = LogisticRegression(max_iter=LOGREG_MAX_ITER, random_state=42)
            clf.fit(X_train, y[train_idx])
            preds_csd[test_idx] = clf.predict(X_test)
        except Exception:
            preds_csd[test_idx] = 0
        try:
            scaler_b = StandardScaler()
            X_train_b = scaler_b.fit_transform(X_base[train_idx])
            X_test_b = scaler_b.transform(X_base[test_idx])
            clf_b = LogisticRegression(max_iter=LOGREG_MAX_ITER, random_state=42)
            clf_b.fit(X_train_b, y[train_idx])
            preds_base[test_idx] = clf_b.predict(X_test_b)
        except Exception:
            preds_base[test_idx] = 0

    return float(f1_score(y, preds_csd, zero_division=0)), float(f1_score(y, preds_base, zero_division=0))


def compute_block6(positive_pairs, block1_results, block2_results):
    """Hypothesis verdict: evaluate 3 success criteria."""
    results = {}

    # SC1: Leading indicator across >= 2 task families and >= 2 models
    leading_tasks, leading_models = set(), set()
    for pair, b1 in zip(positive_pairs, block1_results):
        if b1.get("eval_is_leading", 0) > 0.5:
            leading_tasks.add(pair["task"])
            leading_models.add(pair["smodel"])
    n_task_families = len(leading_tasks)
    n_models = len(leading_models)
    sc1_met = n_task_families >= 2 and n_models >= 2
    results["eval_sc1_met"] = 1.0 if sc1_met else 0.0
    results["eval_sc1_n_task_families"] = float(n_task_families)
    results["eval_sc1_n_models"] = float(n_models)

    # SC2: Variance scaling exponent
    alphas, n_in_range = [], 0
    for b2 in block2_results:
        a = b2.get("eval_bifurcation_alpha", float("nan"))
        if not np.isnan(a):
            alphas.append(a)
            if b2.get("eval_bifurcation_alpha_in_range", 0) > 0.5:
                n_in_range += 1
    sc2_met = n_in_range > 0
    results["eval_sc2_met"] = 1.0 if sc2_met else 0.0
    results["eval_sc2_n_pairs_in_range"] = float(n_in_range)
    results["eval_sc2_mean_alpha"] = float(np.mean(alphas)) if alphas else float("nan")

    # SC3: Classifier improvement >= 15%
    csd_f1s, base_f1s = [], []
    for pair in positive_pairs:
        f1_csd, f1_base = compute_classifier_f1(pair)
        if not np.isnan(f1_csd):
            csd_f1s.append(f1_csd)
            base_f1s.append(f1_base)

    if csd_f1s:
        mean_csd_f1 = float(np.mean(csd_f1s))
        mean_base_f1 = float(np.mean(base_f1s))
        improvement = ((mean_csd_f1 - mean_base_f1) / max(mean_base_f1, 0.01)) * 100
    else:
        mean_csd_f1 = mean_base_f1 = float("nan")
        improvement = float("nan")
    results["eval_sc3_csd_mean_f1"] = mean_csd_f1
    results["eval_sc3_baseline_mean_f1"] = mean_base_f1
    results["eval_sc3_improvement_pct"] = float(improvement) if not np.isnan(improvement) else float("nan")
    sc3_met = not np.isnan(improvement) and improvement >= 15
    results["eval_sc3_met"] = 1.0 if sc3_met else 0.0

    # Overall verdict
    results["eval_hypothesis_confirmed"] = 1.0 if (sc1_met and sc2_met and sc3_met) else 0.0
    results["eval_hypothesis_partially_confirmed"] = 1.0 if sc1_met else 0.0
    all_concurrent = all(b.get("eval_is_leading", 0) < 0.5 for b in block1_results)
    results["eval_hypothesis_disconfirmed"] = 1.0 if all_concurrent else 0.0
    return results


# Run Block 6
print("BLOCK 6: Hypothesis Verdict")
block6_results = compute_block6(positive_pairs, block1_results, block2_results)
print(f"  SC1 (leading indicator): {'MET' if block6_results['eval_sc1_met'] > 0.5 else 'NOT MET'} "
      f"(tasks={block6_results['eval_sc1_n_task_families']:.0f}, models={block6_results['eval_sc1_n_models']:.0f})")
print(f"  SC2 (variance scaling):  {'MET' if block6_results['eval_sc2_met'] > 0.5 else 'NOT MET'} "
      f"(alpha={block6_results['eval_sc2_mean_alpha']:.4f})")
print(f"  SC3 (classifier):        {'MET' if block6_results['eval_sc3_met'] > 0.5 else 'NOT MET'} "
      f"(CSD F1={block6_results['eval_sc3_csd_mean_f1']:.3f}, baseline F1={block6_results['eval_sc3_baseline_mean_f1']:.3f})")
verdict = "CONFIRMED" if block6_results["eval_hypothesis_confirmed"] > 0.5 else \
          "PARTIALLY" if block6_results["eval_hypothesis_partially_confirmed"] > 0.5 else "DISCONFIRMED"
print(f"\n  >>> Hypothesis: {verdict}")

## Block 6: Hypothesis Verdict

Evaluate 3 success criteria:
- **SC1**: CSD flickering is a leading indicator across ≥2 task families and ≥2 models
- **SC2**: Variance scaling exponent α in bifurcation range [-0.7, -0.3]
- **SC3**: CSD-based classifier improves F1 by ≥15% over disagreement-only baseline

In [ ]:
def compute_block5(negative_pairs):
    """Negative control analysis."""
    results = {}
    syl_pairs = [p for p in negative_pairs if p["task"] == "syllogistic"]
    mhop_pairs = [p for p in negative_pairs if p["task"] == "multi_hop"]

    # 5a. Syllogistic (no boundary)
    syl_abs_tau_var, syl_abs_tau_dip = [], []
    syl_sig_count = 0
    for pair in syl_pairs:
        rows = sorted(pair["rows"], key=lambda x: x["difficulty_level"])
        levels = [r["difficulty_level"] for r in rows]
        if len(levels) >= 3:
            var_vals = [r["embedding_variance"] for r in rows]
            tau_v, p_v = stats.kendalltau(levels, var_vals)
            if not np.isnan(tau_v): syl_abs_tau_var.append(abs(tau_v))
            dip_vals = [r["dip_statistic"] for r in rows]
            tau_d, p_d = stats.kendalltau(levels, dip_vals)
            if not np.isnan(tau_d): syl_abs_tau_dip.append(abs(tau_d))
            if (not np.isnan(p_v) and p_v < 0.05) or (not np.isnan(p_d) and p_d < 0.05):
                syl_sig_count += 1

    results["eval_neg_syl_mean_abs_tau_variance"] = float(np.mean(syl_abs_tau_var)) if syl_abs_tau_var else float("nan")
    results["eval_neg_syl_mean_abs_tau_dip"] = float(np.mean(syl_abs_tau_dip)) if syl_abs_tau_dip else float("nan")
    results["eval_neg_syl_fraction_significant"] = float(syl_sig_count / len(syl_pairs)) if syl_pairs else float("nan")

    # 5b. Multi-hop (inverted difficulty)
    mhop_abs_tau_var = []
    mhop_sig_count = 0
    mhop_bimodality_always = True
    for pair in mhop_pairs:
        rows = sorted(pair["rows"], key=lambda x: x["difficulty_level"])
        levels = [r["difficulty_level"] for r in rows]
        if len(levels) >= 3:
            var_vals = [r["embedding_variance"] for r in rows]
            tau_v, p_v = stats.kendalltau(levels, var_vals)
            if not np.isnan(tau_v): mhop_abs_tau_var.append(abs(tau_v))
            if not np.isnan(p_v) and p_v < 0.05: mhop_sig_count += 1
            for r in rows:
                if r["dip_pvalue"] >= 0.05:
                    mhop_bimodality_always = False

    results["eval_neg_mhop_mean_abs_tau_variance"] = float(np.mean(mhop_abs_tau_var)) if mhop_abs_tau_var else float("nan")
    results["eval_neg_mhop_bimodality_always_present"] = 1.0 if mhop_bimodality_always else 0.0
    results["eval_neg_mhop_fraction_significant_trend"] = float(mhop_sig_count / len(mhop_pairs)) if mhop_pairs else float("nan")

    # 5c. Summary
    all_abs_taus = syl_abs_tau_var + syl_abs_tau_dip + mhop_abs_tau_var
    mean_abs_tau = np.mean(all_abs_taus) if all_abs_taus else 1.0
    total_pairs = len(syl_pairs) + len(mhop_pairs)
    total_sig = syl_sig_count + mhop_sig_count
    frac_sig = total_sig / total_pairs if total_pairs > 0 else 1.0
    neg_control_pass = mean_abs_tau < 0.3 and frac_sig < 0.5
    results["eval_negative_control_pass"] = 1.0 if neg_control_pass else 0.0
    return results


# Run Block 5
print("BLOCK 5: Negative Controls")
block5_results = compute_block5(negative_pairs)
print(f"  neg_control_pass={block5_results['eval_negative_control_pass']}")
print(f"  syl_mean_abs_tau_var={block5_results.get('eval_neg_syl_mean_abs_tau_variance', 'nan'):.4f}")
print(f"  mhop_mean_abs_tau_var={block5_results.get('eval_neg_mhop_mean_abs_tau_variance', 'nan'):.4f}")

## Block 5: Negative Controls

Verify that CSD trends are **not** systematically present in negative control pairs:
- **Syllogistic** (no clear difficulty boundary): expect weak/random CSD trends
- **Multi-hop** (inverted difficulty structure): expect no systematic CSD escalation

In [ ]:
def compute_cohens_d(pre_vals, near_vals):
    """Compute Cohen's d effect size."""
    if len(pre_vals) < 2 or len(near_vals) < 2:
        return float("nan")
    n1, n2 = len(pre_vals), len(near_vals)
    s1, s2 = np.std(pre_vals, ddof=1), np.std(near_vals, ddof=1)
    pooled_sd = np.sqrt(((n1 - 1) * s1 ** 2 + (n2 - 1) * s2 ** 2) / (n1 + n2 - 2))
    if pooled_sd < 1e-10:
        return float("nan")
    return float((np.mean(near_vals) - np.mean(pre_vals)) / pooled_sd)


def compute_block4(pair):
    """Effect sizes for a single positive pair."""
    rows = pair["rows"]
    d_star = pair["d_star"]
    pre_zone = [r for r in rows if r["difficulty_level"] < d_star - 3]
    near_zone = [r for r in rows if d_star - 3 <= r["difficulty_level"] <= d_star]

    results = {}
    for field, key in [("embedding_variance", "variance"), ("dip_statistic", "dip"),
                       ("silhouette_k2", "silhouette"), ("bimodality_coefficient", "bc"),
                       ("disagreement_rate", "disagreement")]:
        pre_vals = np.array([r[field] for r in pre_zone if not np.isnan(r[field])])
        near_vals = np.array([r[field] for r in near_zone if not np.isnan(r[field])])
        results[f"eval_cohens_d_{key}"] = compute_cohens_d(pre_vals, near_vals)
    return results


# Run Block 4
print("BLOCK 4: Effect Sizes")
block4_results = []
for i, pair in enumerate(positive_pairs):
    b4 = compute_block4(pair)
    block4_results.append(b4)
    print(f"  [{i+1}/{len(positive_pairs)}] {pair['task']} x {pair['smodel']}: "
          f"d_var={b4['eval_cohens_d_variance']:.3f}, d_dip={b4['eval_cohens_d_dip']:.3f}, "
          f"d_disagr={b4['eval_cohens_d_disagreement']:.3f}")

## Block 4: Effect Sizes (Cohen's d)

Compute Cohen's d effect sizes comparing CSD indicators in pre-boundary vs. near-boundary difficulty zones for each positive pair.

In [ ]:
def compute_block3(positive_pairs, block1_results):
    """Cross-task consistency metrics."""
    n_pairs = len(positive_pairs)
    results = {}

    # 3a. Leading indicator fraction
    n_leading = sum(1 for b in block1_results if b.get("eval_is_leading", 0) > 0.5)
    results["eval_fraction_pairs_with_leading_indicator"] = float(n_leading / n_pairs) if n_pairs > 0 else 0.0

    for ind_key in ["dip", "silhouette", "bc"]:
        n_sig = sum(1 for b in block1_results
                    if not np.isnan(b.get(f"eval_lead_time_{ind_key}", float("nan")))
                    and b.get(f"eval_lead_time_{ind_key}", float("nan")) > 0)
        results[f"eval_fraction_leading_{ind_key}"] = float(n_sig / n_pairs) if n_pairs > 0 else 0.0

    # 3b. Fisher's combined probability test
    for ind_key, ind_name in [("dip", "dip"), ("silhouette", "silhouette"), ("bc", "bimodality_coefficient")]:
        pvalues = []
        for pair, b1 in zip(positive_pairs, block1_results):
            sorted_rows = sorted(pair["rows"], key=lambda x: x["difficulty_level"])
            for r in sorted_rows:
                if r["accuracy"] < 0.8:
                    if ind_name == "dip":
                        p = r["dip_pvalue"]
                    elif ind_name == "silhouette":
                        p = max(0.001, 1.0 - r["silhouette_k2"]) if r["silhouette_k2"] > 0 else 1.0
                    elif ind_name == "bimodality_coefficient":
                        p = max(0.001, 1.0 - r["bimodality_coefficient"]) if r["bimodality_coefficient"] > 0 else 1.0
                    else:
                        p = 1.0
                    if not np.isnan(p) and p > 0:
                        pvalues.append(p)
                    break
        if len(pvalues) >= 2:
            chi2 = -2 * sum(np.log(p) for p in pvalues)
            df = 2 * len(pvalues)
            fisher_p = float(1 - stats.chi2.cdf(chi2, df))
            results[f"eval_fisher_chi2_{ind_key}"] = float(chi2)
            results[f"eval_fisher_pvalue_{ind_key}"] = fisher_p
        else:
            results[f"eval_fisher_chi2_{ind_key}"] = float("nan")
            results[f"eval_fisher_pvalue_{ind_key}"] = float("nan")

    # 3c. Variance trend consistency
    taus = []
    for pair in positive_pairs:
        d_star = pair["d_star"]
        pre = [r for r in pair["rows"] if r["difficulty_level"] <= d_star]
        if len(pre) >= 3:
            levels = [r["difficulty_level"] for r in sorted(pre, key=lambda x: x["difficulty_level"])]
            variances = [r["embedding_variance"] for r in sorted(pre, key=lambda x: x["difficulty_level"])]
            tau, _ = stats.kendalltau(levels, variances)
            if not np.isnan(tau):
                taus.append(tau)
    results["eval_mean_kendall_tau_variance"] = float(np.mean(taus)) if taus else float("nan")
    results["eval_fraction_positive_tau"] = float(sum(1 for t in taus if t > 0) / len(taus)) if taus else float("nan")
    return results


# Run Block 3
print("BLOCK 3: Cross-Task Consistency")
block3_results = compute_block3(positive_pairs, block1_results)
print(f"  fraction_leading={block3_results['eval_fraction_pairs_with_leading_indicator']:.2f}")
print(f"  mean_tau_var={block3_results['eval_mean_kendall_tau_variance']:.4f}")
print(f"  fraction_positive_tau={block3_results['eval_fraction_positive_tau']:.2f}")

## Block 3: Cross-Task Consistency

Evaluate whether CSD leading indicators are consistent across task families using Fisher's combined probability test and Kendall tau variance trends.

In [ ]:
def fit_variance_models(rows, d_star):
    """Fit 4 variance models and compare via AIC/BIC."""
    pre_boundary = sorted([r for r in rows if r["difficulty_level"] <= d_star],
                          key=lambda x: x["difficulty_level"])
    nan_result = {k: float("nan") for k in [
        "eval_best_variance_model", "eval_bifurcation_alpha",
        "eval_bifurcation_alpha_in_range", "eval_bifurcation_r2",
        "eval_aic_bifurcation", "eval_aic_gaussian", "eval_aic_logistic", "eval_aic_null",
        "eval_bic_bifurcation", "eval_bic_gaussian", "eval_bic_logistic", "eval_bic_null",
        "eval_delta_aic_vs_null",
    ]}
    if len(pre_boundary) < 4:
        return nan_result

    d = np.array([r["difficulty_level"] for r in pre_boundary], dtype=float)
    v = np.array([r["embedding_variance"] for r in pre_boundary], dtype=float)
    n = len(d)

    def compute_aic_bic(rss, k, n):
        if rss <= 0 or n <= k:
            return float("inf"), float("inf")
        aic = n * np.log(rss / n) + 2 * k
        bic = n * np.log(rss / n) + k * np.log(n)
        return aic, bic

    results = {}

    # (a) Fold bifurcation power law
    try:
        def power_law(x, A, alpha, C):
            diff = np.maximum(d_star - x, 0.01)
            return A * np.power(diff, alpha) + C
        popt_a, _ = optimize.curve_fit(power_law, d, v, p0=[0.1, -0.5, np.mean(v)],
                                        bounds=([-10, -5, -10], [10, 5, 10]), maxfev=CURVE_FIT_MAXFEV)
        pred_a = power_law(d, *popt_a)
        rss_a = np.sum((v - pred_a) ** 2)
        ss_tot = np.sum((v - np.mean(v)) ** 2)
        r2_a = 1 - rss_a / ss_tot if ss_tot > 0 else 0
        aic_a, bic_a = compute_aic_bic(rss_a, 3, n)
        alpha_val = popt_a[1]
    except Exception:
        rss_a = float("inf"); aic_a = bic_a = float("inf"); r2_a = 0.0; alpha_val = float("nan")

    results["eval_bifurcation_alpha"] = float(alpha_val)
    results["eval_bifurcation_alpha_in_range"] = 1.0 if -0.7 <= alpha_val <= -0.3 else 0.0
    results["eval_bifurcation_r2"] = float(r2_a)
    results["eval_aic_bifurcation"] = float(aic_a)
    results["eval_bic_bifurcation"] = float(bic_a)

    # (b) Gaussian bump
    try:
        def gaussian_bump(x, A, mu, sigma, C):
            return A * np.exp(-((x - mu) ** 2) / (2 * sigma ** 2 + 1e-10)) + C
        popt_b, _ = optimize.curve_fit(gaussian_bump, d, v,
                                        p0=[np.std(v), d_star, (d_star - d[0]) / 2, np.min(v)],
                                        bounds=([-10, d[0] - 5, 0.1, -10], [10, d_star + 5, d_star * 2, 10]),
                                        maxfev=CURVE_FIT_MAXFEV)
        pred_b = gaussian_bump(d, *popt_b)
        rss_b = np.sum((v - pred_b) ** 2)
        aic_b, bic_b = compute_aic_bic(rss_b, 4, n)
    except Exception:
        aic_b = bic_b = float("inf")
    results["eval_aic_gaussian"] = float(aic_b)
    results["eval_bic_gaussian"] = float(bic_b)

    # (c) Logistic S-curve
    try:
        def logistic(x, L, k, d0, C):
            return L / (1 + np.exp(-k * (x - d0))) + C
        popt_c, _ = optimize.curve_fit(logistic, d, v,
                                        p0=[np.ptp(v), 0.5, np.median(d), np.min(v)],
                                        bounds=([-10, -10, d[0] - 5, -10], [10, 10, d_star + 5, 10]),
                                        maxfev=CURVE_FIT_MAXFEV)
        pred_c = logistic(d, *popt_c)
        rss_c = np.sum((v - pred_c) ** 2)
        aic_c, bic_c = compute_aic_bic(rss_c, 4, n)
    except Exception:
        aic_c = bic_c = float("inf")
    results["eval_aic_logistic"] = float(aic_c)
    results["eval_bic_logistic"] = float(bic_c)

    # (d) Flat null
    mean_v = np.mean(v)
    rss_null = np.sum((v - mean_v) ** 2)
    aic_null, bic_null = compute_aic_bic(rss_null, 1, n)
    results["eval_aic_null"] = float(aic_null)
    results["eval_bic_null"] = float(bic_null)

    # Best model by AIC
    models_aic = {"bifurcation": aic_a, "gaussian": aic_b, "logistic": aic_c, "null": aic_null}
    finite_models = {k: v for k, v in models_aic.items() if np.isfinite(v)}
    best = min(finite_models, key=finite_models.get) if finite_models else "null"
    model_encoding = {"bifurcation": 1.0, "gaussian": 2.0, "logistic": 3.0, "null": 4.0}
    results["eval_best_variance_model"] = model_encoding.get(best, 4.0)
    results["eval_delta_aic_vs_null"] = float(aic_a - aic_null) if np.isfinite(aic_a) else float("nan")
    return results


# Run Block 2
print("BLOCK 2: Alternative Variance Models")
block2_results = []
for i, pair in enumerate(positive_pairs):
    b2 = fit_variance_models(pair["rows"], pair["d_star"])
    block2_results.append(b2)
    model_names = {1.0: "bifurcation", 2.0: "gaussian", 3.0: "logistic", 4.0: "null"}
    print(f"  [{i+1}/{len(positive_pairs)}] {pair['task']} x {pair['smodel']}: "
          f"best={model_names.get(b2['eval_best_variance_model'], '?')}, "
          f"alpha={b2['eval_bifurcation_alpha']:.4f}, delta_aic={b2['eval_delta_aic_vs_null']:.2f}")

## Block 2: Alternative Variance Models

Fit 4 variance models to pre-boundary data and compare via AIC/BIC:
- **Bifurcation**: power law `var(d) = A·(d*−d)^α + C`
- **Gaussian bump**: `A·exp(-(d-μ)²/2σ²) + C`
- **Logistic S-curve**: `L/(1+exp(-k(d-d₀))) + C`
- **Null**: flat constant `var(d) = C`

In [ ]:
def find_first_significant_level(rows, d_star, indicator):
    """Find the first difficulty level where an indicator crosses significance threshold."""
    pre_boundary = [r for r in rows if r["difficulty_level"] <= d_star]
    pre_boundary.sort(key=lambda x: x["difficulty_level"])

    thresholds = {
        "dip": lambda r: r["dip_pvalue"] < 0.05,
        "silhouette": lambda r: r["silhouette_k2"] > 0.3,
        "bimodality_coefficient": lambda r: r["bimodality_coefficient"] > 0.555,
        "ashman_d": lambda r: not np.isnan(r["ashman_d"]) and r["ashman_d"] > 2.0,
    }

    if indicator == "embedding_variance":
        levels = [r["difficulty_level"] for r in pre_boundary]
        variances = [r["embedding_variance"] for r in pre_boundary]
        if len(levels) >= 3:
            tau, p = stats.kendalltau(levels, variances)
            if p < 0.05:
                half = max(1, len(pre_boundary) // 2)
                baseline_var = np.median([r["embedding_variance"] for r in pre_boundary[:half]])
                for r in pre_boundary:
                    if r["embedding_variance"] > baseline_var * 1.1:
                        return {"d_lead": r["difficulty_level"], "lead_time": d_star - r["difficulty_level"],
                                "accuracy_at_signal": r["accuracy"], "significant": True, "tau": tau, "p": p}
        return {"d_lead": None, "lead_time": None, "accuracy_at_signal": None, "significant": False}

    check_fn = thresholds.get(indicator)
    if check_fn is None:
        return {"d_lead": None, "lead_time": None, "accuracy_at_signal": None, "significant": False}

    for r in pre_boundary:
        try:
            if check_fn(r):
                return {"d_lead": r["difficulty_level"], "lead_time": d_star - r["difficulty_level"],
                        "accuracy_at_signal": r["accuracy"], "significant": True}
        except (ValueError, TypeError):
            continue
    return {"d_lead": None, "lead_time": None, "accuracy_at_signal": None, "significant": False}


def bootstrap_lead_time_ci(rows, d_star, indicator, B=None):
    """Bootstrap CI on lead_time for a given indicator."""
    if B is None:
        B = BOOTSTRAP_B
    pre_boundary = [r for r in rows if r["difficulty_level"] <= d_star]
    if len(pre_boundary) < 4:
        return {"ci_lower": None, "ci_upper": None, "median": None}

    seeds = RNG.integers(0, 2**31, size=B)
    lead_times = []
    for seed in seeds:
        rng = np.random.default_rng(int(seed))
        n = len(pre_boundary)
        indices = rng.choice(n, size=n, replace=True)
        resampled = sorted([pre_boundary[i] for i in indices], key=lambda x: x["difficulty_level"])
        result = find_first_significant_level(resampled, d_star, indicator)
        lt = result.get("lead_time")
        if lt is not None:
            lead_times.append(lt)

    if len(lead_times) < B * 0.1:
        return {"ci_lower": None, "ci_upper": None, "median": None}

    lead_arr = np.array(lead_times)
    return {"ci_lower": float(np.percentile(lead_arr, 2.5)),
            "ci_upper": float(np.percentile(lead_arr, 97.5)),
            "median": float(np.median(lead_arr))}


def compute_block1(pair):
    """Leading Indicator Tests for a single positive pair."""
    rows = pair["rows"]
    d_star = pair["d_star"]
    results = {}

    indicators = ["dip", "silhouette", "bimodality_coefficient", "ashman_d"]
    lead_times = {}
    earliest_signal_level = None
    earliest_accuracy = None

    for ind in indicators:
        res = find_first_significant_level(rows, d_star, ind)
        key_map = {"dip": "dip", "silhouette": "silhouette", "bimodality_coefficient": "bc", "ashman_d": "ashman_d"}
        short = key_map[ind]
        results[f"eval_lead_time_{short}"] = res["lead_time"] if res["lead_time"] is not None else float("nan")
        lead_times[ind] = res

        if res["significant"] and res["d_lead"] is not None:
            if earliest_signal_level is None or res["d_lead"] < earliest_signal_level:
                earliest_signal_level = res["d_lead"]
                earliest_accuracy = res["accuracy_at_signal"]

    var_res = find_first_significant_level(rows, d_star, "embedding_variance")
    results["eval_variance_trend_significant"] = 1.0 if var_res.get("significant", False) else 0.0
    results["eval_accuracy_at_first_signal"] = earliest_accuracy if earliest_accuracy is not None else float("nan")
    is_leading = earliest_accuracy is not None and earliest_accuracy > 0.8
    results["eval_is_leading"] = 1.0 if is_leading else 0.0

    # Bootstrap CI for best indicator
    best_ind = None
    best_lt = -1
    for ind in indicators:
        lt = lead_times[ind].get("lead_time")
        if lt is not None and lt > best_lt:
            best_lt = lt
            best_ind = ind

    if best_ind is not None and len(rows) >= 10:
        ci = bootstrap_lead_time_ci(rows, d_star, best_ind)
        results["eval_lead_time_ci_lower"] = ci["ci_lower"] if ci["ci_lower"] is not None else float("nan")
        results["eval_lead_time_ci_upper"] = ci["ci_upper"] if ci["ci_upper"] is not None else float("nan")
    else:
        results["eval_lead_time_ci_lower"] = float("nan")
        results["eval_lead_time_ci_upper"] = float("nan")

    return results


# Run Block 1
print("BLOCK 1: Leading Indicator Tests")
block1_results = []
for i, pair in enumerate(positive_pairs):
    b1 = compute_block1(pair)
    block1_results.append(b1)
    print(f"  [{i+1}/{len(positive_pairs)}] {pair['task']} x {pair['smodel']}: "
          f"is_leading={b1['eval_is_leading']}, lead_times: "
          f"dip={b1['eval_lead_time_dip']}, sil={b1['eval_lead_time_silhouette']}, "
          f"bc={b1['eval_lead_time_bc']}")

## Block 1: Leading Indicator Tests

For each positive pair, detect the first difficulty level where CSD indicators (dip test, silhouette, bimodality coefficient) cross significance thresholds **before** the accuracy collapse boundary d*. Bootstrap CIs quantify uncertainty on lead time.

In [ ]:
# Extract unified rows from the evaluation output datasets
def extract_unified_rows(data):
    """Convert evaluation output examples back to unified row format for re-analysis."""
    all_rows = []
    for ds in data["datasets"]:
        for ex in ds["examples"]:
            row = {
                "task_family": ex["metadata_task_family"],
                "model": ex["metadata_model"],
                "difficulty_level": int(ex["metadata_difficulty_level"]),
                "d_star": ex["metadata_d_star"] if ex["metadata_d_star"] != -1 else None,
                "accuracy": float(ex["eval_accuracy_value"]),
                "embedding_variance": float(ex.get("eval_embedding_variance_value", float("nan"))),
                "dip_statistic": float(ex.get("eval_dip_statistic_value", float("nan"))),
                "dip_pvalue": float(ex.get("eval_dip_pvalue_value", float("nan"))),
                "silhouette_k2": float(ex.get("eval_silhouette_value", float("nan"))),
                "bimodality_coefficient": float(ex.get("eval_bimodality_coeff_value", float("nan"))),
                "ashman_d": float("nan"),
                "disagreement_rate": float(ex.get("eval_disagreement_value", float("nan"))),
                "pair_type": ex.get("metadata_pair_type", "positive"),
                "neg_type": ex.get("metadata_neg_type", None),
            }
            # Replace sentinel -999.0 with NaN
            for k in row:
                if isinstance(row[k], float) and row[k] == -999.0:
                    row[k] = float("nan")
            all_rows.append(row)
    return all_rows


def classify_pairs(all_rows):
    """Classify model-task pairs into positive and negative."""
    pair_data = {}
    for r in all_rows:
        key = (r["task_family"], r["model"])
        pair_data.setdefault(key, []).append(r)

    positive_pairs = []
    negative_pairs = []

    for (task, model), rows in pair_data.items():
        d_star = rows[0]["d_star"]
        sorted_rows = sorted(rows, key=lambda x: x["difficulty_level"])
        pair_type = rows[0].get("pair_type", "positive")
        neg_type = rows[0].get("neg_type")

        if pair_type == "negative" or task in ("syllogistic", "multi_hop"):
            neg_t = neg_type or ("no_boundary" if task == "syllogistic" else "inverted")
            negative_pairs.append({"task": task, "model": model, "smodel": model,
                                   "d_star": d_star, "rows": sorted_rows, "neg_type": neg_t})
        elif d_star is not None:
            positive_pairs.append({"task": task, "model": model, "smodel": model,
                                   "d_star": d_star, "rows": sorted_rows})

    return positive_pairs, negative_pairs


all_rows = extract_unified_rows(data)
positive_pairs, negative_pairs = classify_pairs(all_rows)

print(f"Total unified rows: {len(all_rows)}")
print(f"Positive pairs: {len(positive_pairs)}, Negative pairs: {len(negative_pairs)}")
for p in positive_pairs:
    print(f"  + {p['task']} x {p['smodel']} (d*={p['d_star']}, {len(p['rows'])} levels)")
for p in negative_pairs:
    print(f"  - {p['task']} x {p['smodel']} (d*={p['d_star']}, {p['neg_type']}, {len(p['rows'])} levels)")

## Step 0: Extract Unified Rows & Classify Pairs

Convert the per-dataset evaluation output into unified rows, then classify each (task, model) pair as **positive** (has a clear difficulty boundary d*) or **negative** (syllogistic = no boundary, multi-hop = inverted difficulty).

In [ ]:
# ── Tunable parameters ──
BOOTSTRAP_B = 100          # Original: 10000 (bootstrap resampling iterations)
CURVE_FIT_MAXFEV = 5000    # Original: 10000 (max function evaluations for curve_fit)
LOGREG_MAX_ITER = 500      # Original: 1000 (logistic regression max iterations)
RNG = np.random.default_rng(42)

print(f"Config: BOOTSTRAP_B={BOOTSTRAP_B}, CURVE_FIT_MAXFEV={CURVE_FIT_MAXFEV}, LOGREG_MAX_ITER={LOGREG_MAX_ITER}")

## Configuration

Tunable parameters for the evaluation. `BOOTSTRAP_B` controls bootstrap resampling iterations (original: 10000).

In [ ]:
data = load_data()
print(f"Loaded data: {len(data['datasets'])} datasets")
for ds in data['datasets']:
    print(f"  {ds['dataset']}: {len(ds['examples'])} examples")
print(f"Aggregate metrics: {len(data['metrics_agg'])} keys")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-outputs/ai-invention-276cb0-flickering-before-failing-ecological-cri/main/evaluation_iter3_cross_task_csd/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

print("Data loading helper defined")

In [ ]:
import json
import math
import os
import warnings

import numpy as np
from scipy import optimize, stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

print("Imports OK")

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# All packages used are pre-installed on Colab; install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'scikit-learn==1.6.1', 'matplotlib==3.10.0')

# Cross-Task CSD Leading Indicator Evaluation

**Comprehensive evaluation of Critical Slowing Down (CSD) indicators as leading indicators of LLM reasoning collapse.**

This notebook evaluates CSD leading indicators across multiple task families (arithmetic, graph coloring, syllogistic, multi-hop) with positive and negative control pairs. It implements 6 analysis blocks:

1. **Leading Indicator Detection** with bootstrap confidence intervals
2. **Alternative Variance Model Comparison** (AIC/BIC)
3. **Cross-Task Consistency** via Fisher's method
4. **Cohen's d Effect Sizes**
5. **Negative Controls** analysis
6. **3-Criterion Hypothesis Verdict** (SC1, SC2, SC3)